# **Task 2 — DeiT**: does seeing more pixels help?

This week's seminar was about [DeiT](https://proceedings.mlr.press/v139/touvron21a.html): a ViT trained on ImageNet alone, with distillation, and fine-tuned at a higher resolution at the end. The paper reports that going from 224×224 to 384×384 images makes the model more accurate (Table 10). Today you check whether that holds for **our** model and **our** data.

This notebook loads the data and the model and gives you a **fixed evaluation**, so that every group measures its result the same way. Your own work starts after the baseline.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# The task

**Question.** Does seeing more pixels help? Make the model run at 384×384, fine-tune it within the budget, and **answer the question for this model and this data, with evidence**. Then explain *why* you got the result you got, and what they teach you.

*Hint:* be wary of what you compare with, and of how your claim relates to the paper's.

**Fixed:**
- the starting model and the dataset;
- the `evaluate()` function below;
- the fine-tuning budget: **at most 100,000 training images per fine-tuned model** (about 11 passes over the training set);
- write the resizing of the position embeddings yourselves: no ready-made resizing (e.g. timm's `img_size=` argument or `resample_abs_pos_embed`).

Everything else is your choice.

**Bring to the lab:** your code + a short deck with your answer, the evidence for it, and your explanation. If your group defends this week, **every member should be able to explain any piece of the code**.

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need (PyTorch, torchvision, timm) is already installed on Colab.

**Step 1. Data.** [Imagewoof](https://github.com/fastai/imagenette): 10 dog breeds from ImageNet, 9,025 training and 3,929 validation photos, **at full size** (about 1.3 GB, under a minute to download).

In [ ]:
import os, tarfile, time, urllib.request
import torch, timm
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz"
# we use the full-size image sets on purpose, not the smaller "-320" versions
# can you tell why? (in the deck)
if not os.path.exists("imagewoof2/val"):
    urllib.request.urlretrieve(URL, "imagewoof2.tgz")
    with tarfile.open("imagewoof2.tgz") as t:
        t.extractall(filter="data")
for split in ["train", "val"]:
    print(split, sum(len(f) for _, _, f in os.walk(f"imagewoof2/{split}")), "photos")

**Step 2. Model.** A distilled DeiT-S from the paper (12 blocks, 384 dims, 22M parameters, the DeiT-S⚗ of Table 5), trained on ImageNet at 224×224, together with the exact preprocessing it was trained with.

The model predicts one of 1000 ImageNet classes, so we map each Imagewoof folder (e.g. `n02086240` = Shih-Tzu) to its **ImageNet index**: the position of that breed among the model's 1000 outputs. You will need `to_imagenet` for your fine-tuning too.

<details>
<summary>Running the cell below can ask for your HF_TOKEN secret</summary>
Just press "Grant Access", it's automatically generated by Colab
</details>

In [ ]:
model = timm.create_model("deit_small_distilled_patch16_224", pretrained=True).to(device).eval()
cfg = timm.data.resolve_data_config({}, model=model)  # the preprocessing the model was trained with

# ImageFolder labels the 10 folders 0..9, but the model outputs 1000 ImageNet classes:
# to_imagenet[label] translates a folder label into the ImageNet class index
wnids = list(timm.data.ImageNetInfo().label_names())
breeds = datasets.ImageFolder("imagewoof2/val").classes
to_imagenet = torch.tensor([wnids.index(w) for w in breeds], device=device)
print(cfg)
print("folder -> ImageNet index:", dict(zip(breeds, to_imagenet.tolist())))

**Step 3. The fixed evaluation** (DON'T modify it). `evaluate(model, resolution)` resizes the validation photos for the given resolution exactly as the model's own preprocessing does, and returns the **top-1 accuracy over the 10 breeds**: the model's prediction is the highest-scoring of its 10 dog outputs.

In [ ]:
@torch.no_grad()
def evaluate(model, resolution):
    """The fixed evaluation for task 2. Do not change it; report exactly what it prints.

    model      - any model that takes (batch, 3, resolution, resolution) images and returns 1000 logits
    resolution - the input size: 224 for the original model, e.g. 384 for yours
    returns: accuracy - top-1 over the 10 breeds, on the 3,929 Imagewoof validation photos
    """
    model.eval()
    tf = timm.data.create_transform(**dict(cfg, input_size=(3, resolution, resolution)))
    loader = torch.utils.data.DataLoader(datasets.ImageFolder("imagewoof2/val", transform=tf),
                                         batch_size=64, num_workers=2)
    correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
        correct += (logits[:, to_imagenet].argmax(-1) == y).sum().item()  # best of the 10 breeds
        n += len(y)
    return correct / n

**Step 4. Baseline.** Takes about half a minute. You should get **top-1 ≈ 97.20%** at 224. If not, something in the setup went wrong.

In [ ]:
print(f"baseline DeiT-S distilled @224: top-1 {evaluate(model, 224):.2%}")

# Before you start

Think about **what exactly breaks in the model when you pass it higher-resolution images?** What's the simplest, most effective way to fix that?

Training photos are in `imagewoof2/train` (same folder structure as `val`).

# Your solution

Make the model run at 384, fine-tune within the budget, and collect the evidence for your answer. Good luck!

## Step 5. Settings

**First run:** keep `SMOKE_TEST = True`. Every fine-tuning then trains on only 640 images, so the whole notebook runs in a few minutes and you can check that nothing crashes. Smoke outputs go to `outputs_smoke/`, real ones to `outputs/`, so they never mix.

**Real runs:** set `SMOKE_TEST = False` and run this cell again.

What we write to the output folder:
- `results.json`, updated after each evaluation: every model's top-1, plus, for fine-tuned models, the per-epoch training log;
- `<run>.pt`, written once at the end of each fine-tuning run: the fine-tuned weights (about 90 MB each).

These files live on the Colab server and are deleted when it shuts down, so download the output folder when the runs are done.

In [ ]:
# CELL 5.1 | settings, output folder, record()
import copy, json, math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

SMOKE_TEST = True                          # True: quick check that everything runs. False: the real experiment.
BUDGET = 640 if SMOKE_TEST else 100_000    # training images per fine-tuned model (task limit: 100,000)
BATCH_SIZE = 32
LR = 2e-5                                  # CONSTANT LR. TODO: TUNE THE LR AND/OR ADD A SCHEDULE (E.G. WARMUP + COSINE) LATER

OUT_DIR = "outputs_smoke" if SMOKE_TEST else "outputs"     # results.json and the fine-tuned weights go here
RESULTS = os.path.join(OUT_DIR, "results.json")
os.makedirs(OUT_DIR, exist_ok=True)
print("outputs go to:", OUT_DIR)


def record(name, m, resolution, **info):
    """Runs the fixed evaluate() and adds the result to the results file (re-read every time, so runs from
    separate cells, or even separate sessions, all end up in one file)."""
    acc = evaluate(m, resolution)
    results = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
    results[name] = dict(acc=acc, resolution=resolution, **info)
    with open(RESULTS, "w") as f:
        json.dump(results, f, indent=2)
    print(f">>> {name}: top-1 {acc:.2%}  (saved to {RESULTS})")

## Step 6. Make the model run at 384: resize the position embeddings

At 224 the image is a 14×14 grid of 16-px patches (196 tokens, 198 with `[CLS]` and `[DIST]`), and `pos_embed` has exactly 198 learned vectors. At 384 the grid is 24×24 (576 + 2 = 578 tokens), so the table no longer fits.

The fix: treat the 196 patch vectors as a 14×14 "image" with 384 channels, resize it to 24×24 with bicubic interpolation, and leave `[CLS]` and `[DIST]` untouched (they have no position in the grid). All other weights don't depend on the number of tokens, so they work as they are.

In [ ]:
# CELL 6.1 | resize_pos_embed(), at_resolution()
def resize_pos_embed(pos_embed, n_prefix, new_grid):
    """Resizes the position-embedding table: (1, n_prefix + old*old, dim) -> (1, n_prefix + new*new, dim).
    For our model at 384: (1, 198, 384) -> (1, 578, 384)."""
    # 1. Split the table. The first n_prefix = 2 rows belong to [CLS] and [DIST]: they have no place in the
    #    image, so they are kept exactly as they are. The other 196 rows are one vector per patch position.
    prefix, grid = pos_embed[:, :n_prefix], pos_embed[:, n_prefix:]          # (1, 2, 384) and (1, 196, 384)
    old_grid, dim = int(math.sqrt(grid.shape[1])), grid.shape[2]             # 14, 384

    # 2. The 196 rows are stored row by row (rows 0-13 = top row of patches, and so on). Reshape them back
    #    into the 14x14 layout, then move the 384 numbers per position to the front: F.interpolate expects
    #    (batch, channels, height, width). We now have "an image of 14x14 pixels with 384 channels".
    grid = grid.reshape(1, old_grid, old_grid, dim)                          # (1, 14, 14, 384)
    grid = grid.permute(0, 3, 1, 2)                                          # (1, 384, 14, 14)

    # 3. Resize that "image" from 14x14 to 24x24, exactly like resizing a photo: each of the 384 channels is
    #    resized on its own, and bicubic interpolation gives smooth values between neighbouring positions.
    #    (DeiT does the same when it fine-tunes at a higher resolution.)
    grid = F.interpolate(grid, size=(new_grid, new_grid), mode="bicubic", align_corners=False)  # (1, 384, 24, 24)

    # 4. Undo step 2: channels back to the end, then flatten the 24x24 positions into a list of 576 rows.
    #    permute must come before reshape, otherwise numbers from different positions get mixed together.
    grid = grid.permute(0, 2, 3, 1)                                          # (1, 24, 24, 384)
    grid = grid.reshape(1, new_grid * new_grid, dim)                         # (1, 576, 384)

    # 5. Put [CLS] and [DIST] back in front, where the model expects them.
    return torch.cat([prefix, grid], dim=1)                                  # (1, 578, 384)


def at_resolution(src, resolution):
    """Returns a copy of `src` that accepts resolution x resolution images; `src` itself is not modified.
    Only two parts of the model depend on the image size: the position-embedding table and the size check
    in patch_embed. Everything else (the 16x16 patch projection, attention, MLPs, heads) works for any number
    of patches, so those weights are reused unchanged."""
    m = copy.deepcopy(src)                                                   # independent copy: training it never touches `model`
    g = resolution // m.patch_embed.patch_size[0]                            # patches per side: 384 // 16 = 24

    # 1. Replace the position table with the resized one. Wrapping it in nn.Parameter makes it a trainable
    #    weight again, so fine-tuning can correct the interpolated positions. no_grad: this is surgery on the
    #    weights, not a training step, so autograd should not record it.
    with torch.no_grad():
        m.pos_embed = nn.Parameter(resize_pos_embed(m.pos_embed, m.num_prefix_tokens, g))

    # 2. patch_embed remembers the image size the model was built for and raises an error for any other size.
    #    img_size is what that check uses; grid_size and num_patches are updated so the object stays consistent.
    m.patch_embed.img_size = (resolution, resolution)
    m.patch_embed.grid_size = (g, g)
    m.patch_embed.num_patches = g * g
    return m

In [ ]:
# CELL 6.2 | sanity checks (each line must pass silently); also creates m384, used in CELL 7.1
pe = model.pos_embed.detach().clone()
m384 = at_resolution(model, 384).eval()
assert torch.allclose(resize_pos_embed(pe, 2, 14), pe), "14 -> 14 must change nothing"
assert m384.pos_embed.shape == (1, 2 + 24 * 24, 384), m384.pos_embed.shape
assert torch.equal(m384.pos_embed[:, :2], pe[:, :2]), "[CLS] and [DIST] must stay unchanged"
assert torch.equal(model.pos_embed, pe), "the original model must stay untouched"
with torch.no_grad():
    assert m384(torch.randn(2, 3, 384, 384, device=device)).shape == (2, 1000), "384 forward pass failed"
print("all checks passed:", tuple(pe.shape), "->", tuple(m384.pos_embed.shape))

## Step 7. The two models without any training

- **`224_pretrain`**: the original model at 224 (the baseline again, now saved to the JSON).
- **`384_pretrain`**: the same pretrained weights at 384 with resized position embeddings, no training. Shows what the resizing alone does.

In [ ]:
# CELL 7.1 | evaluate 224_pretrain and 384_pretrain (no training)
record("224_pretrain", model, 224, run=None, images_seen=0, train_minutes=0)
record("384_pretrain", m384, 384, run=None, images_seen=0, train_minutes=0)

## Step 8. Fine-tuning

One function used by all four runs, so 224 and 384 get exactly the same recipe and budget; resolution is the only difference. Every run starts from a **fresh copy of the pretrained model**. The loss is computed only over the 10 breed outputs (`to_imagenet`), the same 10 outputs `evaluate()` looks at.

`run` numbers the repeats (1, 2): each run number gives its own, repeatable shuffling and augmentation, so run 1 and run 2 differ, and a 224 and a 384 run with the same number see the photos in the same order.

Nothing is saved during training: if Colab disconnects in the middle of a run, just run that cell again. When a run finishes, its cell saves the result to `results.json` and the weights to `<run>.pt`. To use saved weights later: `m = at_resolution(model, 384); m.load_state_dict(torch.load("outputs/384_fine-tune_run1.pt"))`.

In [ ]:
# CELL 8.1 | train_transform(), finetune()
def train_transform(resolution):
    """A new random version of each photo every time it is loaded (evaluate() instead uses a fixed resize
    + centre crop). Without it every epoch would show the exact same images and the model could memorise them."""
    return transforms.Compose([
        # Pick a random rectangle covering 25-100% of the photo's area (aspect ratio between 3:4 and 4:3),
        # cut it out and resize it to resolution x resolution with bicubic interpolation.
        transforms.RandomResizedCrop(resolution, scale=(0.25, 1.0),
                                     interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(),                                   # mirror left-right with probability 0.5
        transforms.ToTensor(),                                               # PIL image -> (3, H, W) floats in [0, 1]
        transforms.Normalize(cfg["mean"], cfg["std"]),                       # the normalization the model was trained with
    ])


def finetune(resolution, run):
    """Fine-tunes a fresh copy of the pretrained model at `resolution` on BUDGET training images."""
    torch.manual_seed(run)    # PyTorch's random generator: shuffling and random crops/flips come from it -> run 1 != run 2, each repeatable
    m = at_resolution(model, resolution).train()                             # fresh copy; at 224 the resize changes nothing
    train_set = datasets.ImageFolder("imagewoof2/train", transform=train_transform(resolution))
    loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                                         drop_last=True, num_workers=2)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.05)
    scaler = torch.amp.GradScaler()                                          # needed for float16 training
    total_steps = BUDGET // BATCH_SIZE                                       # 3125 steps x 32 = 100,000 images
    step, log, t0 = 0, [], time.time()

    while step < total_steps:                                                # one pass of this loop = one epoch
        losses, correct = [], 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = m(x)[:, to_imagenet]                                # (batch, 10): only the 10 breeds
            loss = F.cross_entropy(logits.float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
            correct += (logits.argmax(-1) == y).sum().item()
            step += 1
            if step == total_steps:                                          # budget reached in the middle of an epoch
                break
        log.append(dict(epoch=len(log) + 1, images_seen=step * BATCH_SIZE, train_loss=float(np.mean(losses)),
                        train_acc=correct / (len(losses) * BATCH_SIZE), minutes=round((time.time() - t0) / 60, 2)))
        e = log[-1]
        print(f"epoch {e['epoch']:>2} | images {e['images_seen']:>7,} | train loss {e['train_loss']:.3f} "
              f"| train acc {e['train_acc']:.1%} | {e['minutes']:.1f} min")

    assert step * BATCH_SIZE <= BUDGET, f"budget exceeded: {step * BATCH_SIZE} images"
    return m.eval(), dict(run=run, images_seen=step * BATCH_SIZE, train_minutes=log[-1]["minutes"], log=log)

### The four runs

Each cell is independent: run them in any order, in one session or several. On a T4, roughly 10 min per 224 run and 20 min per 384 run.

In [ ]:
# CELL 8.2 | RUN 224_fine-tune, run 1: the fair control for 384_fine-tune
name = "224_fine-tune_run1"
m, info = finetune(resolution=224, run=1)
record(name, m, 224, **info)
torch.save(m.state_dict(), os.path.join(OUT_DIR, f"{name}.pt"))          # final weights, saved once
del m; torch.cuda.empty_cache()

In [ ]:
# CELL 8.3 | RUN 224_fine-tune, run 2: same, repeated
name = "224_fine-tune_run2"
m, info = finetune(resolution=224, run=2)
record(name, m, 224, **info)
torch.save(m.state_dict(), os.path.join(OUT_DIR, f"{name}.pt"))          # final weights, saved once
del m; torch.cuda.empty_cache()

In [ ]:
# CELL 8.4 | RUN 384_fine-tune, run 1: the model the question is about
name = "384_fine-tune_run1"
m, info = finetune(resolution=384, run=1)
record(name, m, 384, **info)
torch.save(m.state_dict(), os.path.join(OUT_DIR, f"{name}.pt"))          # final weights, saved once
del m; torch.cuda.empty_cache()

In [ ]:
# CELL 8.5 | RUN 384_fine-tune, run 2: same, repeated
name = "384_fine-tune_run2"
m, info = finetune(resolution=384, run=2)
record(name, m, 384, **info)
torch.save(m.state_dict(), os.path.join(OUT_DIR, f"{name}.pt"))          # final weights, saved once
del m; torch.cuda.empty_cache()

## Step 9. Summary

Reads the results file, so it works after a restart and with only some runs finished.

- **384_fine-tune − 224_fine-tune** is the answer: same fine-tuning, only the resolution differs.
- **224_fine-tune − 224_pretrain** is what fine-tuning alone does.
- **384_fine-tune − 224_pretrain** is the naive comparison: it mixes the two effects.
- **384_pretrain − 224_pretrain** is what the resizing alone does, without training.

In [ ]:
# CELL 9.1 | summary table
R = json.load(open(RESULTS))
print(f"{'model':<22}{'top-1':>8}{'train min':>11}")
for name, r in R.items():
    print(f"{name:<22}{r['acc']:>8.2%}{r['train_minutes']:>11}")


def pts(a, b):  # accuracy of model a minus model b, in percentage points
    return f"{100 * (R[a]['acc'] - R[b]['acc']):+.2f} pts" if a in R and b in R else "(run missing)"


print("\n384_pretrain - 224_pretrain      resizing only, no training:", pts("384_pretrain", "224_pretrain"))
for run in (1, 2):
    print(f"run {run}:")
    print("  384_fine-tune - 224_fine-tune  more pixels, same fine-tuning (THE ANSWER):",
          pts(f"384_fine-tune_run{run}", f"224_fine-tune_run{run}"))
    print("  224_fine-tune - 224_pretrain   fine-tuning only:                          ",
          pts(f"224_fine-tune_run{run}", "224_pretrain"))
    print("  384_fine-tune - 224_pretrain   naive comparison (mixes both effects):     ",
          pts(f"384_fine-tune_run{run}", "224_pretrain"))

print(f"\nfor scale: 1 validation photo = {100 / 3929:.3f} pts; "
      f"standard error of one accuracy at ~97% = {100 * math.sqrt(0.97 * 0.03 / 3929):.2f} pts")